In [9]:
import pandas as pd
from ydata_profiling import ProfileReport
import os
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from pathlib import Path

In [10]:
stock_price_folder = Path("F:/huyle_data_projects/stock_price_prediction/data/for_pretrained_models/prices/")
macro_folder = Path("F:/huyle_data_projects/stock_price_prediction/data/for_pretrained_models/macro/")
output_path = Path("F:/huyle_data_projects/stock_price_prediction/reports/eda_reports/for_pretrained_models/")

In [12]:
def process_stocks_individually():
    print("Processing EDA for individual Stock tickers")
    stock_files = list(stock_price_folder.glob("*.csv"))
    
    for file_path in stock_files:
        ticker_name = file_path.stem
        print(f"Processing Stock EDA for: {ticker_name}")
        
        df = pd.read_csv(file_path)
        
        if 'Date' in df.columns:
            df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%y')
            sort_col = 'Date'
        else:
            sort_col = None
            
        profile = ProfileReport(
            df,
            tsmode=True,
            sortby=sort_col,
            title=f"Stock Data EDA - {ticker_name}",
            explorative=True,
            progress_bar=False
        )
        
        output_file = output_path / f"EDA_Stock_{ticker_name}.html"
        profile.to_file(output_file)
        print(f"Saved: {output_file.name}")

# 3. Process Macro Data (Join by date and generate a combined report)
def process_macro_combined():
    print("Merging and processing EDA for Macro data")
    macro_files = list(macro_folder.glob("*.csv"))
    
    macro_dfs = []
    for file_path in macro_files:
        name = file_path.stem  # e.g., 'gold', 'oil'
        df = pd.read_csv(file_path)
        
        if 'Date' in df.columns and 'Value' in df.columns:
            df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%y')
            
            # Keep only Date and Value columns, rename Value to the indicator's name
            df = df[['Date', 'Value']].rename(columns={'Value': name})
            
            # Set Date as index in preparation for joining
            df.set_index('Date', inplace=True)
            macro_dfs.append(df)
            print(f"Loaded: {name}")
            
    if macro_dfs:
        combined_macro = pd.concat(macro_dfs, axis=1).reset_index()
        combined_macro.sort_values('Date', inplace=True)
        
        print(f"Generating combined Macro report (Total {len(combined_macro.columns) - 1} indicators)...")
        
        profile = ProfileReport(
            combined_macro,
            tsmode=True,
            sortby='Date',
            title="Macro Data EDA - Combined",
            explorative=True,
            progress_bar=False
        )
        
        output_file = output_path / "EDA_Macro_Combined.html"
        profile.to_file(output_file)
        print(f"  -> Saved combined report: {output_file.name}")
    else:
        print("No valid macro data found to merge.")

In [13]:
process_stocks_individually()
process_macro_combined()

Processing EDA for individual Stock tickers
Processing Stock EDA for: FPT


100%|████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:01<00:00,  4.53it/s]


Saved: EDA_Stock_FPT.html
Processing Stock EDA for: HPG


100%|████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:01<00:00,  4.54it/s]


Saved: EDA_Stock_HPG.html
Processing Stock EDA for: MSN


...

Saved: EDA_Stock_VIC.html
Processing Stock EDA for: VNM


100%|████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:01<00:00,  4.54it/s]


Saved: EDA_Stock_VNM.html
Merging and processing EDA for Macro data
Loaded: dxy
Loaded: fed_rate
Loaded: gold
Loaded: oil
Loaded: sp500
Generating combined Macro report (Total 5 indicators)...


100%|████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:01<00:00,  4.79it/s]


  -> Saved combined report: EDA_Macro_Combined.html
